# 01 Look at each source

This notebook opens each source in `data/raw/` on its own.
For each source, it shows the number of rows, the columns, a few rows, and the column that names the lot.
It does not combine any sources. That comes in a later notebook.

Before you run it, download the sources:

```bash
uv run python -m meridian_dashboard_nyc.pull
```

**Words used here.**
- **BBL** is the ten-digit lot number: one borough digit, five block digits and four lot digits. `1000477501` is 120 Broadway.
- **BIN** is the seven-digit building number. One lot can hold several buildings.
- **FAR** is floor area ratio: a building's floor area as a multiple of its lot area.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from meridian_dashboard_nyc import config
from meridian_dashboard_nyc.sources import SOURCES

RAW = config.DATA / "raw"
pd.set_option("display.max_columns", 50)


def newest(source: str) -> Path:
    """Return the folder of the newest complete pull of one source."""
    pulls = sorted(p for p in (RAW / source).iterdir() if not p.name.endswith(".partial"))
    return pulls[-1]


def record(source: str) -> dict:
    """Return what _source.json says about the newest pull of one source."""
    return json.loads((newest(source) / "_source.json").read_text(encoding="utf-8"))


def read(source: str) -> pd.DataFrame:
    """Read the newest pull of one source. Every BBL and BIN stays text, to keep its zeros."""
    path = newest(source) / f"{source}.csv"
    return pd.read_csv(path, dtype={"bbl": str, "bin": str}, low_memory=False)

## What was pulled

One row for each source, from its `_source.json`.
`where` is the filter of the pull. An empty `where` means the pull holds the whole dataset.

In [ ]:
pulled = pd.DataFrame([record(source) for source in SOURCES])
pulled[["source", "portal_id", "rows", "finished_utc", "where"]]

## PLUTO: one row for each tax lot

**Names the lot with:** `bbl`.

PLUTO holds most of the facts that the three developer policies and the risk score check (`docs/V0.md`):

| Column | Meaning | Used for |
|---|---|---|
| `lotarea` | Lot area, in square feet | `lot_area_sqft`, a policy rule |
| `residfar`, `commfar`, `facilfar` | The largest FAR that the zoning allows for homes, for commerce and for community facilities | With `builtfar` and `lotarea`, the unused floor area: a policy rule and a risk factor |
| `builtfar` | The FAR of the buildings on the lot today | The unused floor area |
| `landmark`, `histdist` | A landmark, or a historic district | A deal-breaker and a risk factor |
| `pfirm15_flag` | Inside the 2015 FEMA flood zone | A deal-breaker and a risk factor |
| `edesignum` | An environmental restriction (E-designation) | A risk factor |
| `unitsres` | Homes on the lot | A risk factor: tenants must move |
| `bldgclass` | The building class. Classes that start with `R` are condos. `C6`, `C8`, `D0` and `D4` are co-ops | Condo and co-op lots are flagged and left out of every search |

**Watch out:** PLUTO writes each BBL with decimals, such as `4087860042.00000000`.
The other sources write `4087860042`. Before a join, keep the first ten characters of the PLUTO BBL.

In [ ]:
pluto = read("pluto")
pluto.shape

In [ ]:
facts = [
    "bbl",
    "borough",
    "address",
    "bldgclass",
    "zonedist1",
    "lotarea",
    "bldgarea",
    "numfloors",
    "unitsres",
    "yearbuilt",
    "builtfar",
    "residfar",
    "commfar",
    "facilfar",
    "landmark",
    "histdist",
    "pfirm15_flag",
    "edesignum",
]
pluto[facts].head()

In [ ]:
pluto["borough"].value_counts()

## DOB NOW job filings: one row for each filing

**Names the lot with:** `bbl`. **Names the building with:** `bin`.

The pull keeps three job types, filed since the start date in `config.py`.
A job number that ends in `-I1` is the first filing of a job.
Other endings, such as `-S1` or `-P1`, are later filings for the same job.
Some rows have no `bbl`. Their `borough`, `block` and `lot` columns can build one.

In [ ]:
filings = read("dob_now_job_filings")
filings.shape

In [ ]:
filings["job_type"].value_counts()

In [ ]:
filings[
    [
        "job_filing_number",
        "job_type",
        "filing_date",
        "bbl",
        "house_no",
        "street_name",
        "borough",
        "proposed_no_of_stories",
        "proposed_dwelling_units",
    ]
].head()

## Rolling sales: one row for each sale in the last 12 months

**Names the lot with:** `borough`, `block` and `lot`. This source has no `bbl` column.
To join it to PLUTO later, build the BBL from those three columns.

The pull keeps whole lots and buildings only. It leaves out every building class category with `CONDO` or `COOP` in its name.
`sale_price` is the full price paid. A sale under $100,000 is not a market sale. Code flags it and keeps it.
A pull from before 2026-11-13 still holds the condo and co-op sales.

In [ ]:
sales = read("rolling_sales")
sales.shape

In [ ]:
sales["building_class_category"].value_counts().head(15)

## HPD vacate orders: one row for each order

**Names the lot with:** `bbl`.

The pull keeps open orders only: the orders with an empty `actual_rescind_date`.
A pull from before 2026-11-13 holds every order, so the next cell keeps the open ones.

In [ ]:
vacates = read("hpd_vacate_orders")
vacates.shape

In [ ]:
open_orders = vacates[vacates["actual_rescind_date"].isna()]
open_orders.groupby(["vacate_type", "primary_vacate_reason"]).size()

## How the sources can join later

Each source with a `bbl` column, with an example BBL and the share of rows that have no BBL.
This cell checks each source on its own. It joins nothing.

In [ ]:
with_bbl = {
    "pluto": pluto,
    "dob_now_job_filings": filings,
    "hpd_vacate_orders": vacates,
}
pd.DataFrame(
    {
        "example_bbl": [table["bbl"].dropna().iloc[0] for table in with_bbl.values()],
        "ten_digits": [table["bbl"].str.fullmatch(r"\d{10}").mean() for table in with_bbl.values()],
        "no_bbl": [table["bbl"].isna().mean() for table in with_bbl.values()],
    },
    index=list(with_bbl),
)

| Source | Join it to PLUTO on | Writes the borough as |
|---|---|---|
| `pluto` | The first ten characters of `bbl` | `QN` |
| `dob_now_job_filings` | `bbl`, or a BBL built from `borough`, `block` and `lot` when `bbl` is empty | `Queens` |
| `hpd_vacate_orders` | `bbl` | `QN`, in `boro_short_name` |
| `rolling_sales` | A BBL built from `borough`, `block` and `lot` | `4` |

A BBL built from parts is the borough digit, then the block with zeros in front to five digits, then the lot to four digits.
Queens is borough 4, so block 8786 and lot 42 make `4087860042`.

Nothing here joins the sources. A later notebook does that, and writes its tables to `data/processed/`.